# Algoritmo de Recomendação: ItemKNN

Este notebook introduz o uso do algoritmo itemKNN para recomendação de itens a usuários utilizando a biblioteca [RecBole](https://recbole.io/docs/user_guide/model_intro.html).

## Imports

In [1]:
from pathlib import Path
from datetime import datetime
from logging import getLogger

from recbole.config import Config
from torch.utils.tensorboard import SummaryWriter
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger, get_model, get_trainer

## Carregando arquivo de configuração

A RecBole utiliza basicamente arquivos de configuração para que possamos utilizar os modelos e carregar os datasets. Por isso o arquivo [ml_1m.yaml](ml_1m.yaml) já contém todas as configurações que iremos utilizar para o treinamento do modelo.

In [3]:
config_file = 'ml_1m.yaml'

config = Config(
    model='ItemKNN',
    dataset='ml-1m',
    config_file_list=[config_file]
)

## Inicializações necessárias

Iremos inicializar o logger do RecBole assim como a seed.

In [4]:
init_seed(config['seed'], config['reproducibility'])
init_logger(config)

logger = getLogger()

Visualizando o arquivo de configuração.

In [5]:
logger.info(config)

02 Jun 10:07    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = ./data/atomic/ml-1m
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 4096
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'RS': [8, 1, 1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = False
metrics = ['Recall', 'MRR', 'NDCG', 'Hit', 'MAP', 'Precision', 'GAUC']
topk = [5, 10, 20]
valid_metric = MRR@10
valid_metric_bigger = True
eval_batch_size = 4096
metric_decimal_place = 4

Dataset

## Criando dataset no formato do RecBole

O RecBole aceita datasets em um formato específico, chamado *atomic*, que contém 3 arquivos:

- `dataset.inter`
- `dataset.item`
- `dataset.user`

Portanto, todo dataset novo a ser utilizado deve ser passado para esse formato específico.

> O dataset utilizado será o Movie Lens 1M que já foi transformado para o formato atomic utilizando o [script](convert_ml1m_to_recbole.py).

In [6]:
dataset = create_dataset(config)

logger.info(dataset)

/home/miguel/Documents/UFES/TCC/recsys_fairness/.venv/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/home/miguel/Documents/UFES/TCC/recsys_fairness/.venv/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never wor

Também podemos dividir os dados em treinamento, avaliação e teste.

In [7]:
train_data, valid_data, test_data = data_preparation(config, dataset)

02 Jun 10:07    INFO  [Training]: train_batch_size = [4096] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
02 Jun 10:07    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [8, 1, 1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


## Treino e avaliação do modelo

Primeiro passo será configurar o modelo e o objeto de treino da biblioteca.

In [8]:
model = get_model(config['model'])(config, train_data._dataset).to(config['device'])

trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)

Após isso, podemos iniciar o treinamento.

In [9]:
best_valid_score, best_valid_result = trainer.fit(
    train_data,
    valid_data,
    saved=False,
    show_progress=config['show_progress']
)

Train     0:   0%|                                                          | 0/393 [00:00<?, ?it/s]/home/miguel/Documents/UFES/TCC/recsys_fairness/.venv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
Train     0:   4%|█▋                                              | 14/393 [00:00<00:02, 136.09it/s]:  10%|█████                                           | 41/393 [00:00<00:01, 213.26it/s]:  17%|████████▎                                       | 68/393 [00:00<00:01, 236.62it/s]:  24%|███████████▍                                    | 94/393 [00:00<00:01, 244.51it/s]:  31%|██████████████▍                                | 121/393 [00:00<00:01, 251.77it/s]:  38%|█████████████████▊                             | 149/393 [00:00<00:00, 259.19it/s]:  45%|████████████████████▉                          | 175

Avaliando o modelo no conjunto de teste.

In [10]:
test_result = trainer.evaluate(
    test_data,
    load_best_model=False,
    show_progress=config['show_progress']
)

logger.info(test_result)

Evaluate   :   0%|                                                         | 0/6040 [00:00<?, ?it/s]:   2%|▊                                            | 101/6040 [00:00<00:05, 1008.81it/s]:   3%|█▌                                            | 202/6040 [00:00<00:06, 966.02it/s]:   5%|██▎                                           | 300/6040 [00:00<00:05, 970.63it/s]:   7%|███                                           | 398/6040 [00:00<00:05, 965.60it/s]:   8%|███▊                                          | 495/6040 [00:00<00:05, 952.95it/s]:  10%|████▌                                         | 591/6040 [00:00<00:05, 936.37it/s]:  11%|█████▏                                        | 685/6040 [00:00<00:05, 928.10it/s]:  13%|█████▉                                        | 778/6040 [00:00<00:05, 917.62it/s]:  14%|██████▋                                       | 870/6040 [00:00<00:05, 913.25it/s]:  16%|███████▎                                      | 966/6040 [00:01<00:05, 926.84it/s]:  18%|███